In [ ]:
import pandas as pd
df = pd.read_csv(r"aviationstack_flights_clean.csv")

date_cols = [
    'Departure Scheduled',
    'Departure Actual',
    'Arrival Scheduled',
    'Arrival Actual'
]
for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors='coerce')
df['Departure Delay'] = pd.to_numeric(df['Departure Delay'], errors='coerce')
df['Arrival Delay'] = pd.to_numeric(df['Arrival Delay'], errors='coerce')
mask_dep_delay_missing = (
    df['Departure Delay'].isna() &
    df['Departure Actual'].notna() &
    df['Departure Scheduled'].notna()
)
df.loc[mask_dep_delay_missing, 'Departure Delay'] = (
    (
        df.loc[mask_dep_delay_missing, 'Departure Actual'] -
        df.loc[mask_dep_delay_missing, 'Departure Scheduled']
    ).dt.total_seconds() / 60
)
mask_dep_actual_missing = (
    df['Departure Actual'].isna() &
    df['Departure Scheduled'].notna()
)

df.loc[mask_dep_actual_missing, 'Departure Actual'] = (
    df.loc[mask_dep_actual_missing, 'Departure Scheduled'] +
    pd.to_timedelta(
        df.loc[mask_dep_actual_missing, 'Departure Delay'].fillna(0),
        unit='m'
    )
)
df['Departure Delay'] = df['Departure Delay'].fillna(0)
mask_arr_delay_missing = (
    df['Arrival Delay'].isna() &
    df['Arrival Actual'].notna() &
    df['Arrival Scheduled'].notna()
)

df.loc[mask_arr_delay_missing, 'Arrival Delay'] = (
    (
        df.loc[mask_arr_delay_missing, 'Arrival Actual'] -
        df.loc[mask_arr_delay_missing, 'Arrival Scheduled']
    ).dt.total_seconds() / 60
)
mask_arr_actual_missing = (
    df['Arrival Actual'].isna() &
    df['Arrival Scheduled'].notna()
)
df.loc[mask_arr_actual_missing, 'Arrival Actual'] = (
    df.loc[mask_arr_actual_missing, 'Arrival Scheduled'] +
    pd.to_timedelta(
        df.loc[mask_arr_actual_missing, 'Arrival Delay'].fillna(0),
        unit='m'
    )
)
df['Arrival Delay'] = df['Arrival Delay'].fillna(0)
df['Departure Actual'] = df['Departure Actual'].dt.strftime('%Y-%m-%dT%H:%M:%S+00:00')
df['Arrival Actual'] = df['Arrival Actual'].dt.strftime('%Y-%m-%dT%H:%M:%S+00:00')
df.to_csv("aviationstack_flights_cleaned.csv", index=False)
print("Cleaning Done Successfully")